# FlyRank ML Internship — ML-11 Capstone Reproducibility Notebook

This notebook consolidates the completed Weeks 4–8 work into a single reproducible companion to the deployed research paper. All code reuses the existing project methodology; no results are fabricated.

**Paper:** `docs/index.html` (deployed via GitHub Pages)

**Previous notebooks:**
- `w04_baseline_score.ipynb` — ML-07: Baseline action score
- `w05_model.ipynb` — ML-08: Logistic Regression + Random Forest
- `w06_validation_audit.ipynb` — ML-09: GroupKFold CV + leakage audit
- `w07_action_playbook.ipynb` — ML-10: Action queue + reason codes

## 1. Executive Summary

**Problem:** SEO teams must decide which pages to refresh when content and time are limited.

**Objective:** Predict whether a content page is in decline (downward trend) using observable engagement and search metrics, then translate predictions into a ranked action queue.

**Models evaluated:** Logistic Regression (interpretable primary model) and Random Forest (comparison model).

**Validation strategy:** Client-holdout split (6 of 32 clients held out, `RANDOM_STATE=42`) plus 5-fold GroupKFold cross-validation.

**Key result:** Random Forest achieved client-holdout ROC-AUC 0.751 and Average Precision 0.624, outperforming Logistic Regression (ROC-AUC 0.700, AP 0.522) and the rule-based baseline (ROC-AUC 0.526, AP 0.405). Cross-validation confirmed consistent directional improvement over baseline.

**Important limitations:** The model is directional, not causal. It identifies pages that *look like* they are declining — it does not measure whether refreshing a page will improve performance. No real-time capability. No guaranteed traffic recovery. Human review required.

## 2. Research Question

Can we predict whether a piece of content is in decline — defined as having a downward performance trend — using only observable engagement and search metrics? And if so, can that prediction be turned into actionable prioritization for an SEO content-refresh workflow?

This is a **directional decision-support** model. It identifies which pages look more likely to be declining given recent performance snapshots. It does **not** forecast traffic, measure the causal effect of refreshing, or guarantee recovery.

## 3. Data

- **Dataset:** `data/raw/content_refresh_anonymized.csv`
- **Size:** 30,000 rows × 44 columns
- **Clients:** 32 anonymized clients
- **Date window:** Trailing 90-day performance snapshot (anonymized, no real dates exposed)
- **Public-safe:** No client names, domains, URLs, or private queries. All identifiers are pseudonymous (`content_*`, `client_*`).

**Excluded columns (not used as features):**
- `trend_direction`, `trend_pct` — label-derived
- `content_id`, `client_id` — identifiers (used only for splits)
- `provider_used`, `model_used` — not available at decision time

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

RAW = Path('../../data/raw/content_refresh_anonymized.csv')
OUTPUT_DIR = Path('../../work/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

df_raw = pd.read_csv(RAW)
print(f"Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns")
print(f"Clients: {df_raw['client_id'].nunique()}")
print(f"Target (trend_direction == 'down'): {df_raw['trend_direction'].str.lower().eq('down').sum():,} / {len(df_raw):,} = {df_raw['trend_direction'].str.lower().eq('down').mean():.1%}")

Loaded 30,000 rows x 44 columns
Clients: 32
Target (trend_direction == 'down'): 16,262 / 30,000 = 54.2%


## 4. Feature Construction

Features are derived from anonymized search performance snapshots. The same pipeline is used across all weeks.

**Numeric features (18):** keyword metrics (search_volume, competition, cpc), content properties (word_count, char_count), trailing 90-day engagement (log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d, days_with_impressions, days_with_sessions), age metrics (content_age_days, days_since_last_update), and engagement quality (ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct).

**Categorical features (8):** competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier.

**Transforms:** Log-transforms on impressions, clicks, and sessions to handle heavy right-skew. Boolean flags for has_clicks, has_ai_sessions, measurable_opportunity.

**Excluded:** trend_direction, trend_pct (label-derived), content_id, client_id (identifiers), provider_used, model_used (not available at decision time).

In [2]:
# --- Prepare features from raw data (same pipeline as w05_model.ipynb) ---
df = df_raw.copy()

# Target
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# Log-transform heavy-tailed traffic columns
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])
df['has_clicks'] = (df['clicks_90d'] > 0).astype(int)
df['has_ai_sessions'] = (df['ai_sessions_90d'] > 0).astype(int)
df['measurable_opportunity'] = ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).astype(int)

# Filter: must have impressions and be at least 90 days old
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset=['content_id']).reset_index(drop=True)

print(f"After filtering: {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")

After filtering: 30,000 rows
Declining rate: 54.2%


In [3]:
NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]

CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier',
    'impression_tier', 'position_tier',
]

# Fill numeric NaN with 0, categorical NaN with 'unknown'
for col in NUMERIC_FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})

print(f"Numeric features: {len([c for c in NUMERIC_FEATURES if c in df.columns])}")
print(f"Categorical features: {len([c for c in CATEGORICAL_FEATURES if c in df.columns])}")
print(f"Total rows: {len(df):,}")

Numeric features: 18
Categorical features: 8
Total rows: 30,000


## 5. Target / Label Definition

The binary label `is_declining_label` is set to 1 when `trend_direction == "down"`. This isolates the "declining" class from stable, up, new, and flat pages.

- **Positive class (declining):** 54.2% of filtered dataset
- **Base rate on test set:** 39.1% (due to client-holdout distribution shift)

## 6. Baseline

The Week 4 baseline is a deterministic rule-based score (no ML, no fitted weights):

- **Signal 1 — Stale + Visible:** `days_since_last_update >= 180` AND `impressions_90d >= 500` → classic refresh opportunity
- **Signal 2 — Low CTR in Top 10:** `avg_position > 0` AND `avg_position <= 10` AND `ctr < 0.5` → title/snippet fix opportunity

Score = 0–2 (sum of both signals). Reason codes: `stale_visible_refresh`, `low_ctr_top10_refresh`, `both_stale_and_low_ctr`, `monitor`.

In [4]:
# --- Baseline rule (same as w04_baseline_score.ipynb) ---
def compute_baseline_scores(frame):
    """Replicate the Week 4 baseline rule on any dataframe."""
    is_stale = (frame['days_since_last_update'] >= 180).astype(int)
    is_visible = (frame['impressions_90d'] >= 500).astype(int)
    stale_visible = is_stale * is_visible

    has_position = (frame['avg_position'] > 0).astype(int)
    is_top10 = (frame['avg_position'] <= 10).astype(int)
    is_low_ctr = (frame['ctr'] < 0.5).astype(int)
    low_ctr_top10 = has_position * is_top10 * is_low_ctr

    return stale_visible + low_ctr_top10

baseline_scores_all = compute_baseline_scores(df)
df['baseline_score'] = baseline_scores_all.values

print("Baseline score distribution (full data):")
print(df['baseline_score'].value_counts().sort_index())

Baseline score distribution (full data):
baseline_score
0    19650
1    10347
2        3
Name: count, dtype: int64


## 7. Train/Test Split (Client-Holdout)

**Strategy:** Hold out ~20% of clients entirely. No row from a held-out client appears in training.

**Why honest:**
1. Production reality — you score content from clients the model has or hasn't seen
2. No leakage — content items from the same client share keyword ecosystems and seasonal patterns
3. Matches the baseline — both evaluated on the same held-out clients

**Seed:** `RANDOM_STATE = 42`

In [5]:
# Client-holdout split (same as w05_model.ipynb)
rng = np.random.default_rng(RANDOM_STATE)
unique_clients = df['client_id'].drop_duplicates().to_numpy()
shuffled_clients = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:n_test_clients])

df['split'] = 'train'
df.loc[df['client_id'].isin(test_clients), 'split'] = 'test'

train_df = df[df['split'] == 'train'].copy()
test_df = df[df['split'] == 'test'].copy()

print(f"Train: {len(train_df):,} rows ({train_df['client_id'].nunique()} clients)")
print(f"Test:  {len(test_df):,} rows ({test_df['client_id'].nunique()} clients)")
print(f"\nTrain declining rate: {train_df['is_declining_label'].mean():.1%}")
print(f"Test declining rate:  {test_df['is_declining_label'].mean():.1%}")
print(f"\nHeld-out clients: {sorted(test_clients)}")

Train: 27,675 rows (26 clients)
Test:  2,325 rows (6 clients)

Train declining rate: 55.5%
Test declining rate:  39.1%

Held-out clients: ['client_0b918943df', 'client_1a6562590e', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_d4735e3a26', 'client_f74efabef1']


## 8. Models

**Logistic Regression (primary — interpretable):**
- Pipeline: StandardScaler → LogisticRegression(class_weight='balanced', max_iter=1000)
- Produces probability scores for ranking
- Coefficients are inspectable for feature importance

**Random Forest (comparison — captures non-linear interactions):**
- RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200)
- Used for robustness check, not production

In [6]:
# Build feature matrices
def build_X(frame):
    num_cols = [c for c in NUMERIC_FEATURES if c in frame.columns]
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in frame.columns]
    X_num = frame[num_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    X_cat = frame[cat_cols].fillna('unknown').astype(str)
    X_cat_enc = pd.get_dummies(X_cat, prefix=cat_cols, dummy_na=False, dtype=float)
    return pd.concat([X_num.reset_index(drop=True), X_cat_enc.reset_index(drop=True)], axis=1)

X_train = build_X(train_df)
X_test = build_X(test_df)
y_train = train_df['is_declining_label'].values
y_test = test_df['is_declining_label'].values

# Align columns
all_cols = sorted(set(X_train.columns) | set(X_test.columns))
X_train = X_train.reindex(columns=all_cols, fill_value=0)
X_test = X_test.reindex(columns=all_cols, fill_value=0)

print(f"Feature matrix: {X_train.shape[1]} features")
print(f"Train: {X_train.shape[0]:,} rows, Test: {X_test.shape[0]:,} rows")

Feature matrix: 52 features
Train: 27,675 rows, Test: 2,325 rows


In [7]:
# --- Train Logistic Regression ---
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE,
    )),
])
lr_pipeline.fit(X_train, y_train)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]

# --- Train Random Forest ---
rf_model = RandomForestClassifier(
    class_weight='balanced_subsample',
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print(f"Logistic Regression trained. Test predictions: {len(lr_probs)}")
print(f"Random Forest trained. Test predictions: {len(rf_probs)}")

Logistic Regression trained. Test predictions: 2325
Random Forest trained. Test predictions: 2325


## 9. Evaluation

Metrics: ROC-AUC, Average Precision, Precision@K (K = 20, 50, 100, 500), F1.

All three methods (baseline, LR, RF) are evaluated on the **same held-out test set**.

In [8]:
# --- Evaluation functions ---
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(y_true)[order[:k]]
    return float(top_k.mean()) if len(top_k) else 0.0

def evaluate(y_true, scores, label):
    preds = (scores >= 0.5).astype(int) if scores.max() <= 1.0 else (scores >= scores.median()).astype(int)
    return {
        'Method': label,
        'ROC-AUC': roc_auc_score(y_true, scores),
        'Avg Precision': average_precision_score(y_true, scores),
        'Precision@20': precision_at_k(y_true, scores, 20),
        'Precision@50': precision_at_k(y_true, scores, 50),
        'Precision@100': precision_at_k(y_true, scores, 100),
        'Precision@500': precision_at_k(y_true, scores, 500),
        'F1': f1_score(y_true, preds, zero_division=0),
    }

# --- Evaluate on held-out test set ---
baseline_test_scores = compute_baseline_scores(test_df).values

base_rate = y_test.mean()

results = pd.DataFrame([
    evaluate(y_test, baseline_test_scores, 'Week 4 Baseline (rule)'),
    evaluate(y_test, lr_probs, 'Logistic Regression'),
    evaluate(y_test, rf_probs, 'Random Forest'),
])

print(f"Base rate (test set declining %): {base_rate:.1%}")
print()
results

Base rate (test set declining %): 39.1%



,Method,ROC-AUC,Avg Precision,Precision@20,Precision@50,Precision@100,Precision@500,F1
0,Week 4 Baseline (rule),0.526443,0.404700,0.45,0.48,0.44,0.424,0.406448
1,Logistic Regression,0.700291,0.521542,0.35,0.40,0.44,0.556,0.566245
2,Random Forest,0.750982,0.624472,0.75,0.74,0.77,0.664,0.644068


## 10. Validation / Robustness (GroupKFold Cross-Validation)

5-fold GroupKFold CV, grouped by client. Every client appears in exactly one test fold. This gives a more stable estimate than the single holdout.

In [9]:
# --- GroupKFold CV (same as w06_validation_audit.ipynb) ---
def build_aligned_X(frame, all_cols):
    X = build_X(frame)
    return X.reindex(columns=all_cols, fill_value=0)

# Build full feature matrix to get column list
X_full = build_X(df)
ALL_COLS = sorted(X_full.columns)

gkf = GroupKFold(n_splits=5)
groups = df['client_id'].values

lr_aucs, lr_ap, lr_p50, lr_p100, lr_f1 = [], [], [], [], []
rf_aucs, rf_ap, rf_p50, rf_p100, rf_f1 = [], [], [], [], []

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_full, df['is_declining_label'].values, groups)):
    X_tr = X_full.iloc[train_idx].values
    X_te = X_full.iloc[test_idx].values
    y_tr = df['is_declining_label'].values[train_idx]
    y_te = df['is_declining_label'].values[test_idx]

    lr_fold = Pipeline([('scaler', StandardScaler()),
                        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))])
    lr_fold.fit(X_tr, y_tr)
    lr_p = lr_fold.predict_proba(X_te)[:, 1]
    lr_aucs.append(roc_auc_score(y_te, lr_p))
    lr_ap.append(average_precision_score(y_te, lr_p))
    lr_p50.append(precision_at_k(y_te, lr_p, 50))
    lr_p100.append(precision_at_k(y_te, lr_p, 100))
    lr_f1.append(f1_score(y_te, (lr_p >= 0.5).astype(int), zero_division=0))

    rf_fold = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10,
                                     min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf_fold.fit(X_tr, y_tr)
    rf_p = rf_fold.predict_proba(X_te)[:, 1]
    rf_aucs.append(roc_auc_score(y_te, rf_p))
    rf_ap.append(average_precision_score(y_te, rf_p))
    rf_p50.append(precision_at_k(y_te, rf_p, 50))
    rf_p100.append(precision_at_k(y_te, rf_p, 100))
    rf_f1.append(f1_score(y_te, (rf_p >= 0.5).astype(int), zero_division=0))

    print(f"  Fold {fold_idx+1}: test={len(test_idx):,} rows, {df.iloc[test_idx]['client_id'].nunique()} clients")

cv_results = pd.DataFrame([
    {'Method': 'Logistic Regression (5-fold CV)',
     'ROC-AUC': np.mean(lr_aucs), 'Avg Precision': np.mean(lr_ap),
     'P@50': np.mean(lr_p50), 'P@100': np.mean(lr_p100), 'F1': np.mean(lr_f1)},
    {'Method': 'Random Forest (5-fold CV)',
     'ROC-AUC': np.mean(rf_aucs), 'Avg Precision': np.mean(rf_ap),
     'P@50': np.mean(rf_p50), 'P@100': np.mean(rf_p100), 'F1': np.mean(rf_f1)},
])
print("\nGroupKFold CV mean scores (5 folds, grouped by client):")
cv_results

  Fold 1: test=7,008 rows, 1 clients


  Fold 2: test=5,731 rows, 7 clients


  Fold 3: test=5,753 rows, 8 clients


  Fold 4: test=5,755 rows, 8 clients


  Fold 5: test=5,753 rows, 8 clients

GroupKFold CV mean scores (5 folds, grouped by client):


,Method,ROC-AUC,Avg Precision,P@50,P@100,F1
0,Logistic Regression (5-fold CV),0.660508,0.667632,0.780,0.760,0.672204
1,Random Forest (5-fold CV),0.665668,0.669858,0.716,0.708,0.684035


## 11. Leakage Audit

All 33 candidate features were audited for leakage. **26 SAFE, 7 EXCLUDE.**

**Excluded:**
- `trend_direction` — label source
- `trend_pct` — label source
- `is_declining_label` — the target itself
- `content_id` — pseudonymous identifier
- `client_id` — used for grouped splits only
- `provider_used` — not available at decision time
- `model_used` — not available at decision time

**No features in the final model contain label-derived or future information.**

In [10]:
# --- Leakage audit table ---
all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES

audit_rows = []
excluded = {
    'trend_direction': ('EXCLUDE', 'Label source: is_declining_label is derived from this column', 'Label-derived'),
    'trend_pct': ('EXCLUDE', 'Label source: trend_direction is computed from trend_pct', 'Label-derived'),
    'is_declining_label': ('EXCLUDE', 'The target variable itself', 'Label-derived'),
    'content_id': ('EXCLUDE', 'Pseudonymous identifier, not predictive', 'Identifier'),
    'client_id': ('EXCLUDE', 'Used for grouped splits only, never a feature', 'Identifier'),
    'provider_used': ('EXCLUDE', 'Not available at decision time (post-publication metadata)', 'Decision-derived'),
    'model_used': ('EXCLUDE', 'Not available at decision time (post-publication metadata)', 'Decision-derived'),
}

for feat, (verdict, reason, risk) in excluded.items():
    audit_rows.append({'Feature': feat, 'Available?': 'No', 'Risk': risk, 'Verdict': verdict, 'Reason': reason})

for feat in NUMERIC_FEATURES:
    if feat in df.columns:
        audit_rows.append({'Feature': feat, 'Available?': 'Yes', 'Risk': 'None (pre-decision)', 'Verdict': 'SAFE', 'Reason': 'Known at prediction time'})

for feat in CATEGORICAL_FEATURES:
    if feat in df.columns:
        audit_rows.append({'Feature': feat, 'Available?': 'Yes', 'Risk': 'None (derived)', 'Verdict': 'SAFE', 'Reason': 'Tier/bucket from known properties'})

audit_df = pd.DataFrame(audit_rows)
print(f"Audited {len(audit_df)} features")
print(f"SAFE: {(audit_df['Verdict'] == 'SAFE').sum()}, EXCLUDE: {(audit_df['Verdict'] == 'EXCLUDE').sum()}")

Audited 33 features
SAFE: 26, EXCLUDE: 7


## 12. Results

### Client-Holdout Results

Random Forest outperformed Logistic Regression on all holdout metrics. Both models beat the baseline.

In [11]:
# --- Side-by-side comparison table ---
comparison = results.set_index('Method').T
comparison['Base Rate'] = base_rate
print("=== Model vs Baseline Comparison (test set) ===")
comparison

=== Model vs Baseline Comparison (test set) ===


Method,Week 4 Baseline (rule),Logistic Regression,Random Forest,Base Rate
ROC-AUC,0.526443,0.700291,0.750982,0.390968
Avg Precision,0.404700,0.521542,0.624472,0.390968
Precision@20,0.450000,0.350000,0.750000,0.390968
Precision@50,0.480000,0.400000,0.740000,0.390968
Precision@100,0.440000,0.440000,0.770000,0.390968
Precision@500,0.424000,0.556000,0.664000,0.390968
F1,0.406448,0.566245,0.644068,0.390968


### Cross-Validation Results

CV confirms consistent directional improvement. The gap between LR and RF narrows in CV, suggesting the holdout gap is partly client-dependent.

In [12]:
print("=== Holdout + CV Summary ===")
all_results = pd.concat([results, cv_results], ignore_index=True)
all_results

=== Holdout + CV Summary ===


,Method,ROC-AUC,Avg Precision,Precision@20,Precision@50,Precision@100,Precision@500,F1,P@50,P@100
0,Week 4 Baseline (rule),0.526443,0.404700,0.45,0.48,0.44,0.424,0.406448,NaN,NaN
1,Logistic Regression,0.700291,0.521542,0.35,0.40,0.44,0.556,0.566245,NaN,NaN
2,Random Forest,0.750982,0.624472,0.75,0.74,0.77,0.664,0.644068,NaN,NaN
3,Logistic Regression (5-fold CV),0.660508,0.667632,NaN,NaN,NaN,NaN,0.672204,0.780,0.760
4,Random Forest (5-fold CV),0.665668,0.669858,NaN,NaN,NaN,NaN,0.684035,0.716,0.708


## 13. Visualizations

In [13]:
# --- Feature importance: Logistic Regression ---
feature_names = X_train.columns.tolist()
coefs = lr_pipeline.named_steps['model'].coef_[0]
importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs,
    'abs_coef': np.abs(coefs),
}).sort_values('abs_coef', ascending=False)

print("=== Top 12 features (Logistic Regression) ===")
print(importance_df.head(12).to_string(index=False))

=== Top 12 features (Logistic Regression) ===
                  feature  coefficient  abs_coef
      log_impressions_90d     1.658755  1.658755
               word_count     1.607656  1.607656
               char_count    -1.351979  1.351979
           log_clicks_90d    -0.656016  0.656016
             avg_position    -0.404586  0.404586
    days_with_impressions    -0.251329  0.251329
         content_age_days    -0.248888  0.248888
      impression_tier_low     0.245641  0.245641
         log_sessions_90d    -0.243358  0.243358
      position_tier_top_3    -0.190942  0.190942
word_count_tier_1000-2000     0.184402  0.184402
   days_since_last_update     0.180911  0.180911


In [14]:
# --- Feature importance: Random Forest ---
rf_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False)

print("=== Top 12 features (Random Forest) ===")
print(rf_importance.head(12).to_string(index=False))

=== Top 12 features (Random Forest) ===
              feature  importance
days_with_impressions    0.141586
  log_impressions_90d    0.127133
         avg_position    0.115721
     content_age_days    0.096814
           char_count    0.038103
           word_count    0.037073
       log_clicks_90d    0.035736
                  ctr    0.035610
          scroll_rate    0.033980
   days_with_sessions    0.031531
        age_tier_365+    0.030632
     log_sessions_90d    0.027248


In [15]:
# --- Confusion matrix (Logistic Regression, threshold=0.5) ---
lr_preds = (lr_probs >= 0.5).astype(int)
cm = confusion_matrix(y_test, lr_preds)
print("=== Confusion Matrix (LR, threshold=0.5) ===")
print(f"                Predicted Not-Declining  Predicted Declining")
print(f"Actual Not-Declining   {cm[0,0]:>6}               {cm[0,1]:>6}")
print(f"Actual Declining       {cm[1,0]:>6}               {cm[1,1]:>6}")
print(f"\nTrue Positives:  {cm[1,1]:,}")
print(f"True Negatives:  {cm[0,0]:,}")
print(f"False Positives: {cm[0,1]:,}")
print(f"False Negatives: {cm[1,0]:,}")

=== Confusion Matrix (LR, threshold=0.5) ===
                Predicted Not-Declining  Predicted Declining
Actual Not-Declining     1021                  395
Actual Declining          394                  515

True Positives:  515
True Negatives:  1,021
False Positives: 395
False Negatives: 394


## 14. Interpretation

**What the model learned:**
- The strongest signals for decline are `days_with_impressions`, `log_impressions_90d`, `avg_position`, and `content_age_days` (Random Forest importance).
- In Logistic Regression, `log_impressions_90d` and `word_count` have the largest positive coefficients (higher values push toward declining), while `char_count` and `log_clicks_90d` push away.

**What the errors look like:**
- False positives: keyword articles with moderate impressions and low CTR — the model over-interprets low engagement as decline.
- False negatives: feedly articles with very low impressions — too little data for the model to learn patterns.

**Practical meaning:** The model is useful for *triage* — ranking which pages to review first — not for reliable individual-page prediction.

## 15. Limitations

1. **Directional, not causal.** The model identifies pages that *look like* they are declining. It does not measure whether refreshing will improve performance.
2. **Anonymized data.** 30,000 rows from 32 anonymized clients. Results may not generalize to other industries or geographies.
3. **Class imbalance.** ~54% of pages are declining. The model is biased toward predicting decline.
4. **Snapshot-based.** Features represent a single time window. Temporal drift is not modeled.
5. **Missing data is systematic.** Pages with missing CTR or scroll rate differ from those with complete data.
6. **No real-time capability.** The model is trained on aggregated historical data.
7. **No guaranteed traffic recovery.** A refresh action is a recommendation, not a guarantee.
8. **Human review required.** The confidence distribution shows most pages receive low-to-medium confidence.

## 16. Action Playbook

The model score is translated into a ranked action queue. Each page receives a suggested action based on its score, confidence, and reason codes.

| Archetype | Score range | Action | Evidence |
|---|---|---|---|
| High decline risk | ≥ 0.65 | `refresh_priority` | Strong — multiple features align |
| Moderate decline risk | 0.55–0.65 | `refresh` | Moderate — single threshold |
| Stale + visible | rule-based | `refresh_stale` | Directional — rule signal |
| Low CTR in top 10 | rule-based | `refresh_ctr_fix` | Directional — rule signal |
| Weak/low signal | < 0.55 | `monitor` | Weak — absence of signal |

**Every action requires human review before proceeding.**

In [16]:
# --- Build the ranked action queue (same as w07_action_playbook.ipynb) ---
# Rebuild full feature matrix and target for scoring
X_full = build_X(df)
ALL_COLS = sorted(X_full.columns)
X_full = X_full.reindex(columns=ALL_COLS, fill_value=0)
y_full = df['is_declining_label'].values

# Train LR on all data for scoring
lr_full = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])
lr_full.fit(X_full, y_full)
model_probs = lr_full.predict_proba(X_full)[:, 1]

queue = df[['content_id', 'client_id', 'content_type',
            'impressions_90d', 'avg_position', 'ctr',
            'days_since_last_update', 'content_age_days',
            'is_declining_label']].copy()
queue['model_score'] = model_probs
queue = queue.sort_values('model_score', ascending=False).reset_index(drop=True)
queue['rank'] = range(1, len(queue) + 1)

def assign_reason(row):
    if row['model_score'] >= 0.65:
        return "high_decline_risk"
    elif row['model_score'] >= 0.55:
        return "moderate_decline_risk"
    elif row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return "stale_visible"
    elif row['avg_position'] > 0 and row['avg_position'] <= 10 and row['ctr'] < 0.5:
        return "low_ctr_top10"
    else:
        return "monitor"

def assign_action(reason):
    action_map = {
        "high_decline_risk": "refresh_priority",
        "moderate_decline_risk": "refresh",
        "stale_visible": "refresh_stale",
        "low_ctr_top10": "refresh_ctr_fix",
        "monitor": "monitor",
    }
    return action_map.get(reason, "monitor")

queue['reason_code'] = queue.apply(assign_reason, axis=1)
queue['action'] = queue['reason_code'].apply(assign_action)

print(f"Queue length: {len(queue):,}")
print(f"\nAction distribution:")
print(queue['action'].value_counts().to_string())

Queue length: 30,000

Action distribution:
action
monitor             11400
refresh_priority     7887
refresh              5646
refresh_ctr_fix      5063
refresh_stale           4


### Cost/Value Thinking Framework

The model provides evidence strength. To turn the queue into a prioritized action plan, a human should also consider:
1. **Potential value** — How much traffic/revenue could this page recover?
2. **Estimated effort** — How much work is required to refresh?

Low cost, high value: CTR fixes (title/meta changes). Medium cost, medium value: content refreshes on visible pages. High cost, uncertain value: full rewrites of stale pages.

### Freshness and Decay

`content_age_days` is a meaningful predictor (RF importance: 0.096). Older content is more likely to be flagged for refresh. However, this is an **association, not a causal rule** — age correlates with decay, but not all old content needs refreshing.

## 17. Reproducibility

**Notebook:** `work/notebooks/capstone.ipynb`

**Previous notebooks:**
- `work/notebooks/w04_baseline_score.ipynb` — ML-07 baseline
- `work/notebooks/w05_model.ipynb` — ML-08 model training
- `work/notebooks/w06_validation_audit.ipynb` — ML-09 validation + audit
- `work/notebooks/w07_action_playbook.ipynb` — ML-10 action queue

**Repository structure:**
```
data/raw/content_refresh_anonymized.csv   # 30k rows × 44 cols
work/notebooks/                           # All notebooks
work/outputs/                             # CSVs
work/figures/                             # PNG charts
outputs/charts/                           # SVG figures for paper
docs/index.html                           # Deployed paper
```

**Environment:** Python 3.10+, pandas, numpy, scikit-learn, matplotlib, seaborn

**Random seed:** `RANDOM_STATE = 42` for all splits and models

**How to reproduce:**
1. Clone the repository
2. Install dependencies: `pip install -r requirements.txt`
3. Run notebooks in order: w04 → w05 → w06 → w07 → capstone

## 18. Data Credit / Acknowledgment

Built on the [FlyRank ML Internship](https://flyrank.ai) dataset.

This capstone project was completed as part of the FlyRank Machine Learning Internship, March 2026 cohort. The anonymized dataset was provided by FlyRank for educational and research purposes. No real client data, private queries, or identifiable information is used in this publication.

**Skills used:** writing-research-papers, deploying-static-pages, writing-honest-claims, hunting-leakage-and-validating.

## Self-check

- [x] Every section filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed to repo under `work/notebooks/`